# Template de prova — Extração e Análise de Dados (E&AD)

Prova prática, conteúdo até a Aula 13. Duas regras:

1. **`# [COLE]`** no topo da célula = código de prova. Copie a célula **inteira**, troque o que tem `# <-- troque`. Ela produz só o que o enunciado pede.
2. **`# [NÃO COLE]`** = conferência só para você (números para escrever a resposta, checagens). Rode aqui ou na prova se quiser, mas **não precisa entregar**.

Nenhuma célula precisa ser copiada pela metade. O template roda inteiro de cara com uma base de exemplo (seção 2); na prova, troque `PASTA` na seção 1.

| A questão pede... | Seção | No simulado |
|---|---|---|
| head, shape, ausências | **3** | Q1 |
| limpar: duplicata, texto, número, data, ausente | **4** | Q2 |
| criar taxa com fórmula | **5** | Q2, Q3 |
| tabela por grupo (contagem + mediana), duas categorias | **6** | Q2, Q3, Q5 |
| gráfico de barras / dispersão | **7** | Q3, Q5 |
| por dia, linha no tempo, recorte + top 5 | **8** | Q4 |
| **prever um número** — MAE, R², real × previsto | **9** | Q7 |
| **prever uma categoria** — precisão/recall/F1, matriz, cortes, importâncias | **10** | Q6, Q8 |
| **agrupar sem rótulo** — KMeans | **11** | — |
| extrair de HTML | **12** | — |

Na dúvida sobre qual técnica: `00-fluxograma.md` no repositório.

## 1. Setup (cole uma vez, no topo da prova)

In [ ]:
# [COLE] imports + pasta dos dados + fonte dos gráficos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (mean_absolute_error, r2_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay, silhouette_score)

PASTA = "dados_exemplo"                                             # <-- troque: na prova, "dados"
FONTE = "Fonte: dados sintéticos do Festival ViraBairro (2026)"    # <-- troque pela fonte do enunciado

def fonte(fig):
    fig.text(0.01, -0.02, FONTE, fontsize=8, color="gray")         # fonte dentro da figura, como nas aulas

## 2. Base de exemplo — NÃO COPIE esta seção

Só existe para o template rodar aqui. Cria `dados_exemplo/publicacoes_brutas.csv` (com os defeitos do simulado) e `dados_exemplo/publicacoes_analise.csv` (já tratada). Nunca escreve na pasta `dados/`.

In [ ]:
# [NÃO COLE] gera a base de exemplo
import os, re
rng = np.random.default_rng(42)
n = 300
tema = rng.choice(["Cultura", "Mobilidade", "Saúde", "Trabalho"], n)
formato = rng.choice(["reel", "carrossel", "imagem", "video"], n)
data = pd.Timestamp("2026-08-01") + pd.to_timedelta(rng.integers(0, 56, n), unit="D")
hora = rng.integers(7, 23, n)
perfil = rng.choice(["coletivo", "institucional", "criador"], n, p=[0.45, 0.30, 0.25])
seguidores = np.array([rng.lognormal({"coletivo": 7.5, "institucional": 11, "criador": 9.5}[p], 0.5) for p in perfil]).astype(int) + 100
palavras = ["festival", "bairro", "cultura", "oficina", "gratuito", "vem", "agenda", "hoje", "programa"]
legenda = [" ".join(rng.choice(palavras, {"coletivo": 30, "institucional": 15, "criador": 3}[p] + rng.integers(0, 20)))
           + " 🎉" * rng.integers(0, 5) for p in perfil]
hashtags = [",".join(rng.choice(["virabairro", "cultura", "festival", "rio", "gratis"],
            {"coletivo": 2, "institucional": 0, "criador": 3}[p] + rng.integers(0, 3), replace=False)) for p in perfil]
alcance = (seguidores * rng.uniform(0.3, 2.0, n)).astype(int) + 200
tam = np.array([len(l) for l in legenda])
taxa = (5 + 3 * (formato == "reel") + 1.5 * (formato == "carrossel") + 1.2 * (hora >= 18) - 0.004 * tam
        - 0.6 * np.log10(seguidores) + 1.0 * (tema == "Saúde") + rng.normal(0, 1.2, n)).clip(0.3)
interacoes = (alcance * taxa / 100).round().astype(int)
emoji = re.compile("[\U0001F000-\U0001FAFF☀-➿]")
base = pd.DataFrame({
    "id_publicacao": [f"P{i:04d}" for i in range(n)],
    "data_publicacao": pd.Series(data.strftime("%Y-%m-%d")) + " " + pd.Series(hora).astype(str).str.zfill(2) + ":00",
    "tema": tema, "formato": formato, "hora": hora, "dia_semana": data.dayofweek,
    "seguidores_autor": seguidores, "videos_autor": rng.integers(5, 400, n),
    "legenda": legenda, "hashtags": hashtags, "tamanho_legenda": tam,
    "n_emojis": [len(emoji.findall(l)) for l in legenda],
    "n_hashtags": [0 if h == "" else len(h.split(",")) for h in hashtags],
    "duracao_segundos": np.where(np.isin(formato, ["reel", "video"]), rng.integers(10, 120, n), 0),
    "alcance": alcance, "interacoes": interacoes,
    "compartilhamentos": (interacoes * rng.uniform(0.05, 0.25, n)).astype(int).astype(float),
    "salvamentos": (interacoes * rng.uniform(0.05, 0.30, n)).astype(int).astype(float),
})
base["taxa_engajamento_pct"] = base["interacoes"] / base["alcance"] * 100
os.makedirs("dados_exemplo", exist_ok=True)
base.to_csv("dados_exemplo/publicacoes_analise.csv", index=False)          # a "já tratada"

suja = base.astype({"alcance": object})                                    # a "bruta", com os defeitos do simulado
suja.loc[rng.choice(n, 70, replace=False), "tema"] = suja["tema"].str.lower()
suja.loc[rng.choice(n, 30, replace=False), "tema"] = " " + suja["tema"].str.upper() + " "
idx = rng.choice(n, 50, replace=False)
suja.loc[idx, "data_publicacao"] = pd.to_datetime(suja.loc[idx, "data_publicacao"]).dt.strftime("%d/%m/%Y %H:%M")
suja.loc[rng.choice(n, 6, replace=False), "alcance"] = np.nan
suja.loc[rng.choice(n, 3, replace=False), "alcance"] = 0
suja.loc[rng.choice(n, 2, replace=False), "alcance"] = "sem dado"
suja.loc[rng.choice(n, 8, replace=False), "compartilhamentos"] = np.nan
suja = pd.concat([suja, suja.sample(5, random_state=1)], ignore_index=True)   # duplicatas
suja.to_csv("dados_exemplo/publicacoes_brutas.csv", index=False)
print("base de exemplo criada em dados_exemplo/")

## 3. Diagnóstico da base (Q1)

Separador `;` (export das aulas): `pd.read_csv(arquivo, sep=";")`. Acento quebrado: `encoding="latin-1"`.

In [ ]:
# [COLE] 5 primeiras linhas, dimensões e ausências (só colunas com pelo menos uma)
df_bruto = pd.read_csv(f"{PASTA}/publicacoes_brutas.csv")      # <-- troque o arquivo
display(df_bruto.head())
print(f"Linhas: {df_bruto.shape[0]} | Colunas: {df_bruto.shape[1]}")
ausencias = df_bruto.isna().sum()
ausencias[ausencias > 0]

In [ ]:
# [NÃO COLE] reconhecimento: o que vai precisar ser limpo na seção 4
print(df_bruto.dtypes, "\n")
print("duplicatas por id:", df_bruto.duplicated(subset="id_publicacao").sum())         # <-- troque o id
for col in ["tema", "formato"]:                                                       # <-- colunas de texto
    print(col, "->", sorted(df_bruto[col].dropna().astype(str).unique()))
for col in df_bruto.select_dtypes(exclude="number").columns:
    if pd.to_numeric(df_bruto[col], errors="coerce").notna().mean() > 0.5:
        print(f"'{col}' é texto mas quase tudo é número -> converter na seção 4")

**Para a resposta (limite da base):** "A base cobre apenas [N] publicações de [um festival / X semanas / alguns perfis], com dados [sintéticos / de uma única fonte], então os padrões não podem ser generalizados para outras redes, públicos ou bairros."

## 4. Limpeza (Q2)

Cole a primeira célula só se as datas vierem em formatos misturados. O jeito da Aula 10 (`format="mixed", dayfirst=True`) inverte dia e mês das datas `2026-08-05` no pandas 3, sem dar erro.

In [ ]:
# [COLE] (só se tiver datas em formatos misturados) conversor que acerta em qualquer versão do pandas
def converter_data(serie):
    texto = serie.astype("string").str.strip()
    eh_iso = texto.str.match(r"\d{4}-\d{2}-\d{2}").fillna(False).astype(bool)      # começa com AAAA-MM-DD
    if int(pd.__version__.split(".")[0]) >= 2:
        iso = pd.to_datetime(texto.where(eh_iso), format="ISO8601", errors="coerce")
        outros = pd.to_datetime(texto.where(~eh_iso), format="mixed", dayfirst=True, errors="coerce")
    else:
        iso = pd.to_datetime(texto.where(eh_iso), errors="coerce")
        outros = pd.to_datetime(texto.where(~eh_iso), dayfirst=True, errors="coerce")
    return iso.fillna(outros)

In [ ]:
# [COLE] limpeza: duplicata, texto, números, datas, inválidos e ausentes
df = pd.read_csv(f"{PASTA}/publicacoes_brutas.csv")                   # <-- troque (nova variável)
df = df.drop_duplicates(subset="id_publicacao")                         # <-- coluna identificadora
df["tema"] = df["tema"].str.strip().str.title()                         # <-- coluna de texto
for col in ["alcance", "compartilhamentos", "salvamentos"]:             # <-- colunas que devem ser número
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["data_publicacao"] = converter_data(df["data_publicacao"])           # <-- coluna de data
df = df[df["alcance"] > 0].copy()                                       # denominador ausente, zero ou negativo sai
for col in ["compartilhamentos", "salvamentos"]:                        # <-- parcelas da soma: mediana
    df[col] = df[col].fillna(df[col].median())
df.shape

In [ ]:
# [NÃO COLE] números para escrever as decisões de limpeza
sem_dup = df_bruto.drop_duplicates(subset="id_publicacao")
alc = pd.to_numeric(sem_dup["alcance"], errors="coerce")
print(">>> PARA A RESPOSTA")
print("duplicatas removidas:", len(df_bruto) - len(sem_dup))
print("tema:", df_bruto["tema"].nunique(), "grafias ->", df["tema"].nunique(), sorted(df["tema"].unique()))
print("alcance ausente/zero/negativo/texto descartados:", int((~(alc > 0)).sum()))
print("compartilhamentos preenchidos com a mediana:", int(sem_dup.loc[alc > 0, "compartilhamentos"].isna().sum()))
print("datas não convertidas:", int(df["data_publicacao"].isna().sum()),
      "| período:", df["data_publicacao"].min(), "->", df["data_publicacao"].max(), "(confira com o enunciado)")
print("linhas:", len(df_bruto), "->", len(df))

**Para a resposta:** uma frase por decisão, com o porquê. "Removi X duplicatas por `id_publicacao` para não contar a mesma publicação duas vezes. Padronizei `tema` (espaços e caixa), que tinha Y grafias para 4 temas. Converti as datas tratando os dois formatos separadamente, para não inverter dia e mês. Descartei Z linhas com `alcance` ausente ou não positivo, sem denominador válido para a taxa, e preenchi W ausências de `compartilhamentos` com a mediana, menos sensível a extremos que a média."

## 5. Taxa com fórmula

In [ ]:
# [COLE] monte exatamente a fórmula do enunciado (numerador entre parênteses)
df["taxa_utilidade_pct"] = (df["compartilhamentos"] + df["salvamentos"]) / df["alcance"] * 100     # <-- troque

In [ ]:
# [NÃO COLE] conferência: sem infinito, sem negativo, valores plausíveis
print(df["taxa_utilidade_pct"].describe().round(2))
print("infinitos:", np.isinf(df["taxa_utilidade_pct"]).sum(), "| negativos:", (df["taxa_utilidade_pct"] < 0).sum())

## 6. Tabelas por grupo

Pediu **mediana** → `"median"`; pediu **média** → `"mean"`.

In [ ]:
# [COLE] uma categoria: número de publicações + mediana, da maior para a menor
(df.groupby("tema")["taxa_utilidade_pct"]                     # <-- grupo e coluna
   .agg(publicacoes="count", mediana="median")                # <-- "median" ou "mean"
   .sort_values("mediana", ascending=False)
   .round(2))

In [ ]:
# [COLE] duas categorias: uma linha por combinação, da maior para a menor mediana
analise = pd.read_csv(f"{PASTA}/publicacoes_analise.csv")                          # <-- troque (nova variável)
combos = (analise.groupby(["tema", "formato"])["taxa_engajamento_pct"]             # <-- grupos e coluna
          .agg(publicacoes="count", mediana="median")
          .sort_values("mediana", ascending=False)
          .reset_index())
combos.round(2)

**Para a resposta:** a primeira linha é a melhor combinação; olhe a coluna `publicacoes` — grupo com poucas publicações dá mediana instável (é a limitação para citar). Comparar duas variáveis juntas mostra diferenças que a mediana de uma só esconde.

In [ ]:
# [COLE] (se pedirem por hashtag, Aula 5) uma linha por hashtag com explode
tags = analise.dropna(subset=["hashtags"]).copy()
tags["hashtags"] = tags["hashtags"].str.split(",")
tags = tags.explode("hashtags")
tags["hashtags"] = tags["hashtags"].str.strip().str.lower()
(tags.groupby("hashtags")
     .agg(qtd_posts=("id_publicacao", "count"), engajamento_medio=("taxa_engajamento_pct", "mean"))
     .sort_values("engajamento_medio", ascending=False)
     .round(2))

## 7. Gráficos

Todo gráfico: título que comunica a pergunta + os dois eixos nomeados + fonte dentro da figura.

In [ ]:
# [COLE] barras: comparar categorias
serie = analise.groupby("tema")["taxa_engajamento_pct"].median().sort_values(ascending=False)   # <-- troque

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(serie.index, serie.values, color="#3b6ea5")
ax.set_title("Qual tema tem a maior mediana de engajamento?")     # <-- troque
ax.set_xlabel("Tema")                                              # <-- troque
ax.set_ylabel("Mediana da taxa de engajamento (%)")                # <-- troque
fonte(fig)
fig.tight_layout()
plt.show()

In [ ]:
# [COLE] barras: combinações de duas categorias (usa a tabela combos da seção 6)
rotulos = combos["tema"] + " – " + combos["formato"]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(rotulos, combos["mediana"], color="#27824c")
ax.set_title("Mediana da taxa de engajamento por tema e formato")  # <-- troque
ax.set_xlabel("Combinação tema × formato")
ax.set_ylabel("Mediana da taxa de engajamento (%)")
plt.setp(ax.get_xticklabels(), rotation=60, ha="right")
fonte(fig)
fig.tight_layout()
plt.show()

In [ ]:
# [COLE] (alternativa) dispersão entre duas numéricas
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(analise["seguidores_autor"], analise["taxa_engajamento_pct"], alpha=0.6, color="#27824c")   # <-- troque
ax.set_xscale("log")                                               # tire se não houver valores gigantes
ax.set_title("Seguidores do autor vs. taxa de engajamento")        # <-- troque
ax.set_xlabel("Seguidores do autor (escala log)")
ax.set_ylabel("Taxa de engajamento (%)")
fonte(fig)
fig.tight_layout()
plt.show()

## 8. Por dia e recortes (Q4)

Tabela por dia em ordem **cronológica**. "Média" ≠ "mediana"; "total" = soma.

In [ ]:
# [COLE] data -> dia, tabela diária ordenada
dia = pd.read_csv(f"{PASTA}/publicacoes_analise.csv")                              # <-- troque (nova variável)
dia["data_publicacao"] = pd.to_datetime(dia["data_publicacao"])
dia["dia_publicacao"] = dia["data_publicacao"].dt.date

diario = (dia.groupby("dia_publicacao")
             .agg(publicacoes=("id_publicacao", "count"),
                  engajamento_medio=("taxa_engajamento_pct", "mean"),
                  alcance_total=("alcance", "sum"))
             .sort_index())
diario.round(2)

In [ ]:
# [COLE] linha: evolução por dia
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(diario.index, diario["engajamento_medio"], marker="o", color="#c0392b")
ax.set_title("Taxa média de engajamento por dia")                  # <-- troque
ax.set_xlabel("Dia da publicação")
ax.set_ylabel("Taxa média de engajamento (%)")
ax.tick_params(axis="x", rotation=30)
fonte(fig)
fig.tight_layout()
plt.show()

In [ ]:
# [COLE] recorte + top 5 (cada condição entre parênteses; "a partir de" = >=)
recorte = dia[(dia["formato"] == "reel") & (dia["hora"] >= 18)]                    # <-- troque
recorte.nlargest(5, "taxa_engajamento_pct")[
    ["id_publicacao", "dia_publicacao", "hora", "tema", "taxa_engajamento_pct"]]    # <-- colunas pedidas

In [ ]:
# [NÃO COLE] números para descrever a variação
print(">>> PARA A RESPOSTA")
print(f"{len(diario)} dias | média diária de {diario['engajamento_medio'].min():.2f} a {diario['engajamento_medio'].max():.2f}"
      f" | publicações por dia: {diario['publicacoes'].min()} a {diario['publicacoes'].max()} | recorte: {len(recorte)} posts")

**Para a resposta:** descreva a oscilação sem falar em tendência (período curto) e diga que o recorte **não prova** que horário ou formato causam engajamento — essas peças diferem em tema, legenda e perfil, e não houve comparação controlada.

## 9. Regressão — prever um NÚMERO (Q7)

Features: **só** as permitidas no enunciado. Nada que exista só depois da publicação (alcance, interações, a própria taxa) — isso é vazamento.

In [ ]:
# [COLE] features, alvo e treino/teste
reg = pd.read_csv(f"{PASTA}/publicacoes_analise.csv")                              # <-- troque (nova variável)
FEATURES = ["tema", "formato", "seguidores_autor", "videos_autor", "tamanho_legenda",
            "n_emojis", "n_hashtags", "hora", "dia_semana", "duracao_segundos"]     # <-- as permitidas
X = pd.get_dummies(reg[FEATURES], columns=["tema", "formato"], drop_first=True, dtype=int)   # <-- colunas de texto
y = reg["taxa_engajamento_pct"]                                                     # <-- alvo
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
# [COLE] regressão linear x árvore: tabela de MAE e R²
linear = LinearRegression().fit(X_treino, y_treino)
arvore = DecisionTreeRegressor(max_depth=4, random_state=42).fit(X_treino, y_treino)     # <-- max_depth

tabela_reg = pd.DataFrame([
    {"modelo": "Regressão linear", "MAE": mean_absolute_error(y_teste, linear.predict(X_teste)),
     "R2": r2_score(y_teste, linear.predict(X_teste))},
    {"modelo": "Árvore de regressão", "MAE": mean_absolute_error(y_teste, arvore.predict(X_teste)),
     "R2": r2_score(y_teste, arvore.predict(X_teste))},
]).sort_values("MAE")
tabela_reg.round(3)

In [ ]:
# [COLE] real x previsto do modelo de menor MAE, com a linha previsão = real
melhor = linear if tabela_reg.iloc[0]["modelo"] == "Regressão linear" else arvore
y_prev = melhor.predict(X_teste)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_teste, y_prev, alpha=0.6, color="#3b6ea5")
lo, hi = min(y_teste.min(), y_prev.min()), max(y_teste.max(), y_prev.max())
ax.plot([lo, hi], [lo, hi], "--", color="gray", label="previsão = valor real")
ax.set_title(f"Valores reais vs. previstos — {tabela_reg.iloc[0]['modelo']}")
ax.set_xlabel("Taxa de engajamento real (%)")                       # <-- troque
ax.set_ylabel("Taxa de engajamento prevista (%)")                   # <-- troque
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# [NÃO COLE] modelo bobo (a média do treino) — o professor usa como referência; cole só se pedirem baseline
mae_bobo = mean_absolute_error(y_teste, np.full(len(y_teste), y_treino.mean()))
print(">>> PARA A RESPOSTA")
print(f"escolhido: {tabela_reg.iloc[0]['modelo']} | MAE {tabela_reg.iloc[0]['MAE']:.2f} | R² {tabela_reg.iloc[0]['R2']:.3f}"
      f" | modelo bobo: MAE {mae_bobo:.2f}")

**Para a resposta:** "É **regressão** porque o alvo é numérico contínuo. O **MAE** é o erro médio absoluto na unidade do alvo: em média a previsão erra X. Escolhi [modelo], de menor MAE. Limitação: associação nesta base, não causa; fatores não medidos também explicam o engajamento." R² negativo = pior que chutar a média (reporte, não é erro seu). R² ≈ 1 de primeira = vazamento.

## 10. Classificação — prever uma CATEGORIA (Q6, Q8)

Os 6 da lista do simulado: regressão logística, árvore, Random Forest, `ExtraTreesClassifier`, `AdaBoostClassifier`, `GaussianNB` (estes dois últimos **não** aceitam `class_weight`). O template usa os três primeiros.

In [ ]:
# [COLE] rótulo pelo percentil 75, features e treino/teste estratificado
clf = pd.read_csv(f"{PASTA}/publicacoes_analise.csv")                              # <-- troque (nova variável)
corte = clf["taxa_engajamento_pct"].quantile(0.75)                                  # <-- percentil
clf["mereceu_divulgacao_adicional"] = (clf["taxa_engajamento_pct"] > corte).astype(int)
FEATURES = ["tema", "formato", "seguidores_autor", "videos_autor", "tamanho_legenda",
            "n_emojis", "n_hashtags", "hora", "dia_semana", "duracao_segundos"]     # <-- as permitidas (NUNCA a taxa)
X = pd.get_dummies(clf[FEATURES], columns=["tema", "formato"], drop_first=True, dtype=int)
y = clf["mereceu_divulgacao_adicional"]
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
# [COLE] três classificadores: precisão, recall e F1 no teste
modelos = {
    "Regressão logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Árvore de classificação": DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight="balanced"),
}
linhas = []
for nome, m in modelos.items():
    p = m.fit(X_treino, y_treino).predict(X_teste)
    linhas.append({"modelo": nome,
                   "precisao": precision_score(y_teste, p, zero_division=0),
                   "recall": recall_score(y_teste, p, zero_division=0),
                   "f1": f1_score(y_teste, p, zero_division=0)})
tabela_clf = pd.DataFrame(linhas).sort_values("f1", ascending=False)
tabela_clf.round(3)

In [ ]:
# [COLE] matriz de confusão do modelo com maior F1
melhor_nome = tabela_clf.iloc[0]["modelo"]
ConfusionMatrixDisplay(confusion_matrix(y_teste, modelos[melhor_nome].predict(X_teste)),
                       display_labels=["Não", "Sim"]).plot(cmap="Blues")
plt.title(f"Matriz de confusão — {melhor_nome}")
plt.show()

In [ ]:
# [COLE] cortes 0,50 e 0,30 na regressão logística JÁ ajustada, no mesmo teste
prob = modelos["Regressão logística"].predict_proba(X_teste)[:, 1]
tabela_cortes = pd.DataFrame([
    {"corte": c,
     "precisao": precision_score(y_teste, (prob >= c).astype(int), zero_division=0),
     "recall": recall_score(y_teste, (prob >= c).astype(int), zero_division=0),
     "f1": f1_score(y_teste, (prob >= c).astype(int), zero_division=0)}
    for c in [0.50, 0.30]                                                           # <-- cortes
])
tabela_cortes.round(3)

In [ ]:
# [COLE] (Q6) árvore max_depth=4 balanceada: 5 maiores importâncias, tabela + barra horizontal
arvore_clf = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced").fit(X_treino, y_treino)
top5 = pd.Series(arvore_clf.feature_importances_, index=X_treino.columns).sort_values(ascending=False).head(5)
display(top5.round(4).to_frame("importancia"))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top5.index[::-1], top5.values[::-1], color="#c0392b")   # invertido: a maior fica no topo
ax.set_title("As 5 características mais usadas pela árvore")
ax.set_xlabel("Importância")
ax.set_ylabel("Característica")
fig.tight_layout()
plt.show()

In [ ]:
# [NÃO COLE] números para escrever sobre erros e cortes
vn, fp, fn, vp = confusion_matrix(y_teste, modelos[melhor_nome].predict(X_teste)).ravel()
a, b = tabela_cortes.iloc[0], tabela_cortes.iloc[-1]
print(">>> PARA A RESPOSTA")
print(f"classe 1: {y.mean():.0%} das publicações | maior F1: {melhor_nome}")
print(f"matriz: VN={vn} FP={fp} (indicou sem merecer) FN={fn} (deixou passar) VP={vp}")
print(f"corte {a['corte']:.2f} -> {b['corte']:.2f}: recall {a['recall']:.2f} -> {b['recall']:.2f}, "
      f"precisão {a['precisao']:.2f} -> {b['precisao']:.2f}")
print(f"acurácia de quem responde sempre 'não': {1 - y_teste.mean():.2f} (por isso acurácia engana)")

**Para a resposta:**
- *Por que classificação:* alvo categórico (0/1). Regressão estimaria um número, mas a decisão é binária; clusterização agrupa sem rótulo, e aqui o rótulo existe.
- *FP* = divulgar o que não merecia (desperdício de espaço escasso). *FN* = não divulgar o que merecia (oportunidade perdida).
- *Corte:* baixar sobe o recall e derruba a precisão. Poucos espaços → precisão; custo alto de deixar passar → recall. Cite os números.
- *Importância alta ≠ causa:* é o que a árvore mais usou **nesta base**; mudar a composição da base muda o ranking.
- A logística fica sem `class_weight` porque, como na Aula 12, a classe rara é compensada pelo corte.

## 11. Clusterização — agrupar SEM rótulo (Aula 13)

Padronizar é obrigatório. Coluna de cauda longa (seguidores, alcance) leva log antes, senão meia dúzia de contas gigantes vira um cluster sozinha.

In [ ]:
# [COLE] padronizar + cotovelo e silhueta para escolher k
clu = pd.read_csv(f"{PASTA}/publicacoes_analise.csv")                              # <-- troque (nova variável)
COLS = ["seguidores_autor", "tamanho_legenda", "n_hashtags", "taxa_engajamento_pct"]   # <-- troque
COLS_LOG = ["seguidores_autor"]                                                     # <-- cauda longa ([] se nenhuma)
entrada = clu[COLS].dropna().copy()
entrada[COLS_LOG] = np.log1p(entrada[COLS_LOG])
X_pad = StandardScaler().fit_transform(entrada)

ks = list(range(2, 9))
inercias = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_pad).inertia_ for k in ks]
silhuetas = [silhouette_score(X_pad, KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_pad)) for k in ks]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(ks, inercias, marker="o"); ax1.set_title("Cotovelo: inércia por k"); ax1.set_xlabel("k"); ax1.set_ylabel("inércia")
ax2.plot(ks, silhuetas, marker="o", color="#c0392b"); ax2.set_title("Silhueta por k"); ax2.set_xlabel("k"); ax2.set_ylabel("silhueta")
fig.tight_layout()
plt.show()

In [ ]:
# [COLE] ajustar com o k escolhido e perfilar (médias nos valores originais)
K = 3                                                                               # <-- k escolhido
perfil = clu.loc[entrada.index, COLS].copy()
perfil["cluster"] = KMeans(n_clusters=K, n_init=10, random_state=42).fit_predict(X_pad)
resumo_clusters = perfil.groupby("cluster")[COLS].mean()
resumo_clusters["tamanho"] = perfil["cluster"].value_counts().sort_index()
resumo_clusters.round(2)

In [ ]:
# [NÃO COLE] silhueta por k e média geral, para escolher k e nomear os grupos
print(">>> PARA A RESPOSTA")
print("silhueta:", {k: round(s, 3) for k, s in zip(ks, silhuetas)}, "| maior em k =", ks[int(np.argmax(silhuetas))])
print("média geral:", perfil[COLS].mean().round(2).to_dict())

**Para a resposta:** "Sem variável-alvo, por isso clusterização. Padronizei (e usei log em seguidores) porque o KMeans mede distância. Escolhi k = X pelo cotovelo e pela maior silhueta (Y). O grupo A reúne [descrição com números]; o B, [...]." Os números dos clusters são etiquetas, não ordem.

## 12. Extrair de HTML (Aula 8)

In [ ]:
# [COLE] achar o bloco que se repete e, dentro dele, cada campo
from bs4 import BeautifulSoup

html = """<ul><li class="post"><h2 class="titulo-post">Oficina de grafite</h2><a class="link-post" href="/p/1">ver</a></li>
<li class="post"><h2 class="titulo-post">Roda de samba</h2></li></ul>"""          # <-- troque: open("arquivo.html", encoding="utf-8").read()

sopa = BeautifulSoup(html, "html.parser")
linhas = []
for bloco in sopa.find_all("li", {"class": "post"}):                               # <-- o bloco
    titulo = bloco.find("h2", {"class": "titulo-post"})                            # <-- cada campo
    link = bloco.find("a", {"class": "link-post"})
    linhas.append({"titulo": titulo.get_text(strip=True) if titulo else None,
                   "link": link["href"] if link else None})
coletado = pd.DataFrame(linhas)
coletado

---

## Lembretes finais

1. Toda questão tem **resposta escrita**. Use as células `[NÃO COLE]` para achar os números.
2. **Nunca afirme causa**: "associado a", "tende a", "nesta base".
3. **Mediana ≠ média**; **utilidade ≠ engajamento**. Leia a palavra.
4. **Gráfico**: título + eixos + fonte.
5. **Datas**: confira o período depois de converter.
6. **Limite de frases** da resposta.
7. Antes de enviar: nome, matrícula, todas as células executadas.